In [1]:
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()
database_url = os.getenv('DATABASE_URL')
engine = create_engine(database_url)
print("db 연결 성공!")

db 연결 성공!


## eligibilities 자격요건 table

In [2]:
query = '''
SELECT nct_id, gender, minimum_age, maximum_age, healthy_volunteers
FROM ctgov.eligibilities;
'''
df_eligibilities = pd.read_sql(query, engine)
df_eligibilities.head()

,nct_id,gender,minimum_age,maximum_age,healthy_volunteers
0,NCT03185689,ALL,18 Years,NaN,False
1,NCT03148730,ALL,60 Years,NaN,False
2,NCT01734655,FEMALE,18 Years,40 Years,True
3,NCT02545725,ALL,18 Years,90 Years,False
4,NCT03882801,ALL,18 Years,75 Years,True


In [3]:
print(df_eligibilities.shape)

(574110, 5)


In [4]:
# study_type이 interventional(중재연구) 만 필터링
load_path = r'C:\dev\clinicaltrials-study\data\processed\studies_clean.csv'
df_studies = pd.read_csv(load_path)
df_eligibilities = df_eligibilities.merge(df_studies[['nct_id', 'study_type']], on='nct_id', how='left')
df_eligibilities = df_eligibilities[df_eligibilities['study_type']=='INTERVENTIONAL']
df_eligibilities.drop(columns=['study_type'], inplace=True)
print(df_eligibilities.shape)

(426519, 5)


In [5]:
df_eligibilities.info()

<class 'pandas.DataFrame'>
Index: 426519 entries, 0 to 574109
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   nct_id              426519 non-null  str   
 1   gender              426334 non-null  str   
 2   minimum_age         408567 non-null  str   
 3   maximum_age         240665 non-null  str   
 4   healthy_volunteers  425787 non-null  object
dtypes: object(1), str(4)
memory usage: 19.5+ MB


In [6]:
# 컬럼 값 전처리
df_eligibilities['gender'].value_counts()

gender
ALL       366298
FEMALE     41131
MALE       18905
Name: count, dtype: int64

In [7]:
df_eligibilities['minimum_age'].value_counts()

minimum_age
18 Years     272654
20 Years      16552
40 Years       9394
21 Years       8790
50 Years       7386
              ...  
259 Days          1
26 Months         1
65 Days           1
24 Days           1
94 Months         1
Name: count, Length: 315, dtype: int64

In [8]:
df_eligibilities['maximum_age'].value_counts()

maximum_age
65 Years     31808
75 Years     25833
80 Years     22344
70 Years     19322
60 Years     13478
             ...  
4383 Days        1
200 Years        1
293 Days         1
191 Hours        1
98 Months        1
Name: count, Length: 471, dtype: int64

In [9]:
df_eligibilities['healthy_volunteers'].value_counts()

healthy_volunteers
False    309707
True     116080
Name: count, dtype: int64

In [10]:
# 1. gender컬럼, 2. healthy_volunteers 컬럼
df_eligibilities['gender']=df_eligibilities['gender'].str.lower().fillna('all')
df_eligibilities['healthy_volunteers'] = df_eligibilities['healthy_volunteers'].map({True:1, False:0}).fillna(-1)

In [11]:
df_eligibilities['gender'].value_counts()

gender
all       366483
female     41131
male       18905
Name: count, dtype: int64

In [12]:
df_eligibilities['healthy_volunteers'].value_counts()

healthy_volunteers
 0.0    309707
 1.0    116080
-1.0       732
Name: count, dtype: int64

In [15]:
# 3. maximum_age, mimimum_age 컬럼
# 나이 단위 환산하여 표준화 시키기
# 나이 타입 str -> numeric으로 변환
import numpy as np
def convert_age_to_years(age_str):
    if pd.isnull(age_str):
        return np.nan
    val, unit = age_str.lower().split()  # 문자열을 소문자로 변환 후 공백 기준으로 숫자와 단위 나눔
    val = float(val)   # 나이를 실수로
    if 'year' in unit:
        return round(val, 2)       # 그대로 반환, 소수점 둘째자리 반올림
    elif 'month' in unit:
        return round(val/ 12, 2)   # 월: 12개월로 나눔
    elif 'week' in unit:
        return round(val/52, 2)    # 주: 52주로 나눔
    elif 'day' in unit:
        return round(val/365, 2)   # 일: 365일로 나눔
    else:
        return np.nan

# 표준화된 함수 컬럼 적용하기
df_eligibilities['minimum_age'] = df_eligibilities['minimum_age'].apply(convert_age_to_years)
df_eligibilities['maximum_age'] = df_eligibilities['maximum_age'].apply(convert_age_to_years)

In [16]:
df_eligibilities['minimum_age'].value_counts()

minimum_age
18.00     272655
20.00      16552
40.00       9394
21.00       8790
50.00       7386
           ...  
0.49           1
0.47           1
100.00         1
2.17           1
7.83           1
Name: count, Length: 202, dtype: int64

In [17]:
df_eligibilities['maximum_age'].value_counts()

maximum_age
65.00     31808
75.00     25833
80.00     22344
70.00     19322
60.00     13478
          ...  
76.25         1
12.01         1
200.00        1
0.80          1
8.17          1
Name: count, Length: 307, dtype: int64

In [18]:
# 연령대 그룹으로 그룹핑 (추론)
def age_group(min_age, max_age):
    if pd.isna(min_age) and pd.isna(max_age):
        return 'unknown'
    if pd.isna(min_age):   # 18세 이상(상한없음) : min 없고, max만 있음. 최대나이가 결정
        return 'child' if max_age < 18 else 'adult' if max_age <=65 else 'senior'
    if pd.isna(max_age):   # 65세 이하(하한없음) : max 없고, min만 있음. 최소나이가 결정
        return 'mixed' if min_age < 18 else 'adult' if min_age <=65 else 'senior'
    if max_age < 18:   
        return 'child'
    if min_age >= 18 and max_age <= 65:   
        return 'adult' 
    if min_age > 65:
        return 'senior'   
    return 'mixed'     

In [19]:
df_eligibilities['age_group'] = df_eligibilities.apply(lambda row: age_group(row['minimum_age'], row['maximum_age']), axis=1)
df_eligibilities.drop(columns=['minimum_age', 'maximum_age'], inplace=True)
df_eligibilities['age_group'].value_counts()

age_group
adult      259596
mixed      124498
child       26470
unknown     13725
senior       2230
Name: count, dtype: int64

In [20]:
# 원핫 인코딩 컬럼: gender, age_group
df_eligibilities = pd.get_dummies(df_eligibilities, columns=['gender', 'age_group'], prefix=['elig_gender', 'elig_age'], dtype=int)
# nct_id 그룹으로 정렬
df_eligibilities = df_eligibilities.groupby('nct_id').max().reset_index()
df_eligibilities.info()

<class 'pandas.DataFrame'>
RangeIndex: 426519 entries, 0 to 426518
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   nct_id              426519 non-null  str    
 1   healthy_volunteers  426519 non-null  float64
 2   elig_gender_all     426519 non-null  int64  
 3   elig_gender_female  426519 non-null  int64  
 4   elig_gender_male    426519 non-null  int64  
 5   elig_age_adult      426519 non-null  int64  
 6   elig_age_child      426519 non-null  int64  
 7   elig_age_mixed      426519 non-null  int64  
 8   elig_age_senior     426519 non-null  int64  
 9   elig_age_unknown    426519 non-null  int64  
dtypes: float64(1), int64(8), str(1)
memory usage: 32.5 MB


In [21]:
# 최종 데이터셋 저장
target_path = r'C:\dev\clinicaltrials-study\data\processed'
file_name = 'eligibilities_clean.csv'
full_save_path = os.path.join(target_path, file_name)
df_eligibilities.to_csv(full_save_path, index=False)